# 01 — Exploratory Data Analysis: Geodata Indonesia

**Project:** Spatial Base System (P1)  
**Tujuan:** Validasi dan eksplorasi layer administrasi Indonesia hasil processing GADM  
**Input:** `data/spatial/processed/*.gpkg`  

---

## Checklist EDA ini
- [ ] Load semua layer
- [ ] Cek jumlah fitur per level
- [ ] Validasi geometri
- [ ] Distribusi ukuran wilayah
- [ ] Temuan anomali
- [ ] Generate summary untuk README

In [ ]:
# Setup
import sys
sys.path.insert(0, '../src')

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from spatial_utils import (
    load_indonesia_levels,
    validate_geodata,
    print_validation_report,
    get_indonesia_summary,
    quick_map,
    plot_indonesia,
)

PROCESSED_DIR = '../data/spatial/processed/'
print('✓ Setup selesai')

## 1. Load Semua Layer

In [ ]:
layers = load_indonesia_levels(
    data_dir=PROCESSED_DIR,
    levels=['provinsi', 'kabupaten', 'kecamatan']
)

# Quick summary
summary = get_indonesia_summary(layers)
print('\n── Summary ──')
print(summary.to_string(index=False))

## 2. Validasi Detail per Layer

In [ ]:
for level, gdf in layers.items():
    report = validate_geodata(gdf, level=level)
    print_validation_report(report)

## 3. Cek Kolom dan Sample Data

In [ ]:
for level, gdf in layers.items():
    print(f'\n── {level.upper()} ──')
    print(f'Columns: {list(gdf.columns)}')
    display(gdf.drop(columns='geometry').head(3))

## 4. Distribusi Luas Wilayah

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
level_titles = {'provinsi': 'Provinsi', 'kabupaten': 'Kabupaten/Kota', 'kecamatan': 'Kecamatan'}

for i, (level, gdf) in enumerate(layers.items()):
    if 'area_km2' not in gdf.columns:
        continue
    ax = axes[i]
    gdf['area_km2'].plot.hist(
        bins=50, ax=ax, color='#2196F3', edgecolor='white', alpha=0.85
    )
    ax.set_title(f'Distribusi Luas — {level_titles.get(level, level)}', fontweight='bold')
    ax.set_xlabel('Luas (km²)')
    ax.set_ylabel('Frekuensi')

    # Statistik deskriptif
    stats = gdf['area_km2'].describe()
    ax.axvline(stats['mean'], color='red', linestyle='--', alpha=0.7, label=f"Mean: {stats['mean']:.0f}")
    ax.axvline(stats['50%'], color='orange', linestyle='--', alpha=0.7, label=f"Median: {stats['50%']:.0f}")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('../assets/screenshots/distribusi_luas_wilayah.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n✓ Saved ke assets/screenshots/')

## 5. Top 10 & Bottom 10 Wilayah Terluas

In [ ]:
if 'kabupaten' in layers and 'area_km2' in layers['kabupaten'].columns:
    kab = layers['kabupaten'].drop(columns='geometry')

    print('── 10 Kabupaten Terluas ──')
    top10 = kab.nlargest(10, 'area_km2')[['nama_kab', 'nama_prov', 'area_km2']]
    display(top10.reset_index(drop=True))

    print('\n── 10 Kabupaten Terkecil ──')
    bottom10 = kab.nsmallest(10, 'area_km2')[['nama_kab', 'nama_prov', 'area_km2']]
    display(bottom10.reset_index(drop=True))

## 6. Peta Statis

In [ ]:
if 'provinsi' in layers:
    fig = plot_indonesia(
        layers['provinsi'],
        title='Provinsi Indonesia — Batas Administrasi',
    )
    plt.show()

if 'kabupaten' in layers and 'area_km2' in layers['kabupaten'].columns:
    fig = plot_indonesia(
        layers['kabupaten'],
        column='area_km2',
        title='Kabupaten/Kota — Luas Wilayah (km²)',
        cmap='YlOrRd',
    )
    plt.show()

## 7. Peta Interaktif

In [ ]:
if 'provinsi' in layers:
    m = quick_map(
        layers['provinsi'],
        title='Indonesia — Batas Provinsi',
        tooltip_cols=['nama_prov', 'area_km2'],
        zoom_start=5,
    )
    display(m)

## 8. Temuan & Catatan Metodologi

> **Isi bagian ini setelah kamu menjalankan notebook dan menemukan anomali atau insight**

### Temuan
- [ ] ...
- [ ] ...

### Anomali yang Perlu Ditindaklanjuti
- [ ] ...

### Keputusan Metodologi
- [ ] ...

---
*Salin temuan penting ke `docs/methodology.md`*